# Chapter 16 &mdash; Hard to Solve, Easy to Check

**Concept 1 of the Chapter 16 decomposition:** *Hard to Solve, Easy to Check: TSP and the Idea of a Certificate*

Checking a claimed tour is polynomial; finding one seems to need factorial work.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-Hard-To-Solve-Easy-To-Check/Concept-Hard-To-Solve-Easy-To-Check.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The **Travelling Salesperson Problem**: visit $n$ cities, return home, keep the total
under a budget.

* **Checking** a claimed tour: add up $n$ numbers and compare. Linear.
* **Finding** one: the obvious method inspects $(n-1)!$ orderings. For $n=49$ that is
  more tours than atoms you can count.

That gap &mdash; **easy to check, apparently hard to find** &mdash; is the whole subject. A
claimed solution you can check quickly is a **certificate**, and the class of problems
with polynomial-time-checkable certificates is **NP**.

Nobody has proved the gap is real. "$P$ vs $NP$" asks exactly whether easy-to-check
implies easy-to-find.

## 2. Definitions

### A TSP instance, a checker, and a brute-force solver

In [ ]:
import itertools, random
def make_instance(n, seed=7):
    random.seed(seed)
    pts = [(random.randint(0, 100), random.randint(0, 100)) for _ in range(n)]
    def d(i, j):
        (a, b), (c_, e) = pts[i], pts[j]
        return round(((a-c_)**2 + (b-e)**2) ** 0.5)
    return pts, [[d(i, j) for j in range(n)] for i in range(n)]

def tour_cost(D, tour):
    return sum(D[tour[i]][tour[(i+1) % len(tour)]] for i in range(len(tour)))

def check_tour(D, tour, budget):
    # the CERTIFICATE checker: linear in the number of cities
    n = len(D)
    return sorted(tour) == list(range(n)) and tour_cost(D, tour) <= budget

def brute_force(D):
    n = len(D)
    best, bt = None, None
    for perm in itertools.permutations(range(1, n)):
        t = (0,) + perm
        c_ = tour_cost(D, t)
        if best is None or c_ < best: best, bt = c_, t
    return best, bt

### Counting the work each side does

In [ ]:
from math import factorial
def search_space(n): return factorial(n - 1)
def check_work(n):   return n

## 3. Tests

Checking is cheap.

In [ ]:
pts, D = make_instance(8)
best, tour = brute_force(D)
print("an optimal tour :", tour, " cost", best)
print("checking it against budget %d : %s" % (best, check_tour(D, tour, best)))
print("checking against budget %d : %s" % (best - 1, check_tour(D, tour, best - 1)))
assert check_tour(D, tour, best) and not check_tour(D, tour, best - 1)

Finding is not.

In [ ]:
print("%-6s %-18s %s" % ("n", "tours to inspect", "additions to check one"))
for n in [5, 8, 12, 20, 49]:
    print("%-6d %-18s %d" % (n, format(search_space(n), ','), check_work(n)))
assert search_space(49) > 10 ** 60

A **certificate** is the claimed tour; anyone can verify it.

In [ ]:
claims = [tour, tuple(range(8)), (0, 2, 1, 3, 4, 5, 6, 7)]
for cl in claims:
    print("  %-26s cost %-5d within budget %d? %s"
          % (str(cl), tour_cost(D, cl), best, check_tour(D, cl, best)))
print("\nNo search is needed to check.  That asymmetry defines NP.")

The gap, measured on real timings at small $n$.

In [ ]:
import time
for n in [6, 8, 9]:
    pts, D = make_instance(n)
    t0 = time.time(); b, t = brute_force(D); t1 = time.time()
    t2 = time.time(); check_tour(D, t, b); t3 = time.time()
    print("  n=%d : search %.4fs, check %.7fs, ratio %.0fx"
          % (n, t1-t0, t3-t2, (t1-t0) / max(t3-t2, 1e-9)))

And the open question.

In [ ]:
print("P   : problems SOLVABLE in polynomial time")
print("NP  : problems whose solutions are CHECKABLE in polynomial time")
print()
print("P subset of NP : obvious -- solve it, then you have the certificate")
print("NP subset of P : unknown, and worth a million dollars")

## 4. Exercises


1. Give the certificate for "this graph has a Hamiltonian cycle". How long is it?
2. Is "this graph has NO Hamiltonian cycle" in NP? What would the certificate be?
3. Why does the checker have to be polynomial in the **input** size, not the certificate's?

In [ ]:
# Your work for the exercises above.